# 01 — One shared dataset for all four screening arms

**Purpose:** validate one prepared citation cohort and one separate human-reference CSV for binary and hybrid NOUL across JEV-9B and JEV-27B v1.1. A different question design does not require a second conversion or dataset. Verify that the four source selections produce identical ordered study IDs, normalized state checksums and reference decisions.

Default input is the existing prepared conversion selected by `configs/evaluations/comparison.yaml`. The workbook, JSONL, labels, saved pilot manifests and inference results are not changed. This notebook makes no API calls. It reports validation, missingness and a read-only deterministic sample preview; export is optional and off.

Model states contain only `study_id`, `title`, `abstract`. Human Inclusion Status, ITEM_ID and psychiatric-context annotations remain separate. Use `DATA_MODE = "demo"` for the two invented teaching records; demo mode cannot establish real four-arm equivalence.

## 1. Load the shared inputs through all four profiles

Use the project `.venv` kernel. Prepared mode reads the binary/hybrid source profiles explicitly, independent of the setup default approach. Both models in both designs must resolve to the same ordered citation content and human reference. A mismatch blocks preparation rather than creating separate experimental datasets.

No `.env`, backend/client, API call or new conversion is needed. If files are missing, obtain the private prepared bundle and review its YAML paths; do not replace it with invented records to produce research results.

In [ ]:
from jev_eval.comparison import comparison_setup_table, load_shared_comparison_inputs
from jev_eval.environment import find_project_root, initialize_notebook

DATA_MODE = "prepared"
root = find_project_root()
shared_inputs = None
if DATA_MODE == "prepared":
    shared_inputs = load_shared_comparison_inputs(root)
    project = shared_inputs.projects[0]
elif DATA_MODE == "demo":
    project = initialize_notebook(load_env=False)
else:
    raise RuntimeError('Choose DATA_MODE="prepared" or "demo" explicitly.')
print("Data mode:", DATA_MODE)
print(
    "Prepared source:" if shared_inputs is not None else "Invented source:",
    project.evaluation.dataset.path,
)

## 2. Inspect validation counts and human-reference linkage

Prepared mode validates field names/types, unique original IDs and citation missingness in every arm. Human reference rows are validated separately, including explicit label mapping, duplicate/missing reference IDs and missing/shared ITEM_ID values. Study ID is the join key; ITEM_ID is not used to merge citations.

The code reports aggregate counts without displaying citation text, study identifiers or row-level human labels. Any difference in mapped human reference decisions across arms blocks loading.

In [ ]:
from jev_eval.data import load_states, summarize_states, validate_states

if shared_inputs is not None:
    validation = shared_inputs.validation
    rows = [state.model_dump() for state in shared_inputs.states]
    display(shared_inputs.references.summary())
else:
    rows = load_states(project.evaluation.dataset, project.root)
    validation = validate_states(rows)
display(summarize_states(validation))

## 3. Review problems without changing decisions

Duplicate IDs, invalid field types or extra input fields block preparation. Human screening decisions must never enter model inputs. Missing abstracts can leave usable title-only studies. Entirely textless studies remain in the cohort and manual-review queue; the runner records them as unscored without sending a request.

Missing labels or ITEM_ID values are reported separately from citation missingness. A technical error or missing score is not an eligibility exclusion. Workbook discrepancy findings were recorded during the archived conversion without automatic adjudication. This notebook reuses that conversion and does not reopen or edit the workbook.

In [ ]:
import pandas as pd

pd.DataFrame(validation.issues, columns=["row", "field", "severity", "issue"])

In [ ]:
states = validation.require_valid()
print(f"Validated {len(states)} states; no records silently excluded.")

## 4. Preview one reproducible sample for all arms

This optional preview uses a single seed and the citation-only pool with at least a title or abstract. All four arms receive the same ordered sample hash; no arm-specific random sampling is introduced. Up to ten studies are previewed, with fewer if the available teaching/test dataset is smaller.

The preview writes no frozen pilot manifest and does not replace existing pilot samples or the original development/tuning membership. It is not new inference and cannot change previously observed results.

In [ ]:
from jev_eval.data import dataset_fingerprint, select_sample

available = [state for state in states if state.title.strip() or state.abstract.strip()]
seed = project.evaluation.seed
if shared_inputs is not None and shared_inputs.table.seed.nunique() != 1:
    raise RuntimeError("Review different sampling seeds before defining a shared pilot preview.")
sample = select_sample(available, min(10, len(available)), seed)
print(f"Read-only sample preview: {len(sample)} studies")
print("Complete dataset SHA-256:", dataset_fingerprint(states))
print("Preview SHA-256:", dataset_fingerprint(sample))

## 5. Optional text inspection — may display private data

The next cell displays citation text only if `privacy.display_record_text` is true in notebook settings. Leave it false for routine shared workflows. If you choose to inspect text, it may be saved in notebook outputs by Jupyter; clear all outputs before committing.

In [ ]:
if project.workflow.privacy.display_record_text:
    display(summarize_states(validation, include_text=True))
else:
    print("Citation text display is disabled; aggregate checks are available above.")

## 6. Optional export — writes a file

**File-writing warning:** enabling the next cell creates a CSV under ignored `results/`. It does not modify the dataset. Enable `output.allow_writes` in YAML and set `ALLOW_WRITES` below to true. Existing files are never overwritten by this export function.

In [ ]:
from jev_eval.reporting import export_table

ALLOW_WRITES = False
if ALLOW_WRITES:
    export_table(
        summarize_states(validation),
        project,
        "results/data-validation/summary.csv",
        allow_writes=True,
    )
else:
    print("Export skipped; no files written.")

## 7. Keep questions and results separate, without duplicating data

Proceed to notebook 02 for synthetic request examples, then 03 for the four saved arms. Distinct task definitions and model deployments require distinct run identities and output directories. They do not require duplicate copies of the prepared studies or human labels.

Existing binary and hybrid overall wording differs and must remain labelled as such. The identical input cohort does not remove that methodological limitation.

## 8. Verify the shared cohort across binary/hybrid × both models

The table below is the main four-arm preparation check. It reports original ID ordering equivalence, normalized state and source-file checksums, reference checksum/mapping, missingness and sample identity. One prepared cohort and one human-reference file are retained in memory; separate copies are not exported for each arm.

Do not create a replacement random development/tuning split. The protocol's original membership and cohort discrepancy must be resolved before formal tuning. The planned 2,000 unseen studies are not loaded here. In demo mode, configuration is shown but real cohort equivalence is explicitly unverified.

In [ ]:
if shared_inputs is not None:
    input_equivalence = shared_inputs.table.assign(
        preview_records=len(sample),
        preview_sha256=dataset_fingerprint(sample),
    )
    display(input_equivalence)
    print("Verified: four arms, one ordered citation cohort and one mapped human reference.")
else:
    input_equivalence = None
    display(comparison_setup_table(project.root))
    print("DEMO ONLY: actual four-arm data equivalence has not been checked.")
print("No workbook conversion, sample freezing, inference or input/result modification performed.")